In [1]:
!pip install -q langchain langchain-groq langchain-community chromadb sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 93.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 74.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71

In [2]:
try:
    import langchain
    import chromadb
    from sentence_transformers import SentenceTransformer
    from langchain_groq import ChatGroq
    print("✅ langchain installed!")
    print("✅ chromadb installed!")
    print("✅ sentence_transformers installed!")
    print("✅ langchain_groq installed!")
    print("\n🎉 Everything is ready!")
except ImportError as e:
    print(f"❌ Missing library: {e}")


✅ langchain installed!
✅ chromadb installed!
✅ sentence_transformers installed!
✅ langchain_groq installed!

🎉 Everything is ready!


upload file


In [5]:
from google.colab import files
import io
!pip install python-docx pypdf
from docx import Document

print("📄 Step 1: Upload your CV file (.docx, .pdf, or .txt)")
uploaded_cv = files.upload()

cv_filename = list(uploaded_cv.keys())[0]
cv_content = uploaded_cv[cv_filename]

# Read CV based on file type
if cv_filename.endswith(".txt"):
    cv_text = cv_content.decode("utf-8")
elif cv_filename.endswith(".pdf"):
    from pypdf import PdfReader
    reader = PdfReader(io.BytesIO(cv_content))
    cv_text = ""
    for page in reader.pages:
        cv_text += page.extract_text()
elif cv_filename.endswith(".docx"):
    doc = Document(io.BytesIO(cv_content))
    cv_text = ""
    for paragraph in doc.paragraphs:
        cv_text += paragraph.text + "\n"

print(f"✅ CV uploaded: {cv_filename}")
print(f"\n📝 CV Preview:\n{cv_text[:300]}...")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.0/349.0 kB 6.5 MB/s eta 0:00:00
📄 Step 1: Upload your CV file (.docx, .pdf, or .txt)


Saving Sonaina_Ilyas_Resume.pdf to Sonaina_Ilyas_Resume.pdf
✅ CV uploaded: Sonaina_Ilyas_Resume.pdf

📝 CV Preview:
Sonaina Ilyas 
Currently in Kazakhstan | sonainailyas005@gmail.com | +7 771 795 7991 | +92 324 
9752902 
Professional Summary 
Data-driven professional with a Master's degree in Computer Science and a Google Data 
Analytics Professional Certificate. Skilled in data cleaning, analysis, and visualizat...


upload job description


In [6]:
print("📋 Step 2: Upload Job Description file (.docx, .pdf, or .txt)")
uploaded_jd = files.upload()

jd_filename = list(uploaded_jd.keys())[0]
jd_content = uploaded_jd[jd_filename]

# Read job description based on file type
if jd_filename.endswith(".txt"):
    jd_text = jd_content.decode("utf-8")
elif jd_filename.endswith(".pdf"):
    from pypdf import PdfReader
    reader = PdfReader(io.BytesIO(jd_content))
    jd_text = ""
    for page in reader.pages:
        jd_text += page.extract_text()
elif jd_filename.endswith(".docx"):
    doc = Document(io.BytesIO(jd_content))
    jd_text = ""
    for paragraph in doc.paragraphs:
        jd_text += paragraph.text + "\n"

print(f"✅ Job Description uploaded: {jd_filename}")
print(f"\n📝 Job Description Preview:\n{jd_text[:300]}...")

📋 Step 2: Upload Job Description file (.docx, .pdf, or .txt)


Saving jobDes.txt to jobDes.txt
✅ Job Description uploaded: jobDes.txt

📝 Job Description Preview:
Job Title: Data Analyst
Company: ABC Company

Requirements:
- 2+ years experience in data analysis
- Python and SQL skills
- Experience with Tableau or Power BI
- Strong communication skills
- AI and Machine Learning knowledge preferred

Responsibilities:
- Analyze large datasets
- Creat...


Converting to chunks than embedding and storing in verctor database

In [8]:
!pip install langchain-text-splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

print("⚙️ Setting up vector database...")

# Step 1 - Split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

cv_chunks = text_splitter.create_documents(
    texts=[cv_text],
    metadatas=[{"source": "cv"}]
)

jd_chunks = text_splitter.create_documents(
    texts=[jd_text],
    metadatas=[{"source": "job_description"}]
)

all_chunks = cv_chunks + jd_chunks

print(f"✅ CV split into {len(cv_chunks)} chunks")
print(f"✅ Job description split into {len(jd_chunks)} chunks")
print(f"✅ Total chunks: {len(all_chunks)}")

# Step 2 - Create embeddings
print("\n⚙️ Creating embeddings...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Step 3 - Store in vector database
print("⚙️ Storing in vector database...")
vectorstore = Chroma.from_documents(
    documents=all_chunks,
    embedding=embeddings
)

print("\n🎉 Vector database ready!")

/tmp/ipykernel_22528/2396118786.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma
/tmp/ipykernel_22528/2396118786.py:32: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


⚙️ Setting up vector database...
✅ CV split into 4 chunks
✅ Job description split into 1 chunks
✅ Total chunks: 5

⚙️ Creating embeddings...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

⚙️ Storing in vector database...

🎉 Vector database ready!


retrieving data from vector

In [9]:
# Create retriever from vector database
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

print("✅ Retriever ready!")

# Test it
test_query = "What are the key skills and experience?"
retrieved_docs = retriever.invoke(test_query)

print(f"\n🔍 Test retrieval for: '{test_query}'")
print(f"📄 Retrieved {len(retrieved_docs)} chunks:\n")

for i, doc in enumerate(retrieved_docs):
    print(f"Chunk {i+1} (from {doc.metadata['source']}):")
    print(doc.page_content[:200])
    print("-" * 30)

✅ Retriever ready!

🔍 Test retrieval for: 'What are the key skills and experience?'
📄 Retrieved 3 chunks:

Chunk 1 (from cv):
drive improvement. 
Core Skills 
Data Cleaning & Visualization | SQL, Excel, Google Sheets | Python (basic) | MS Office (Word, 
Excel, PowerPoint) | Communication & Reporting | Problem-Solving & Time 
------------------------------
Chunk 2 (from job_description):
Job Title: Data Analyst
Company: ABC Company

Requirements:
- 2+ years experience in data analysis
- Python and SQL skills
- Experience with Tableau or Power BI
- Strong communication skills
-
------------------------------
Chunk 3 (from cv):
Database systems. 
 Designed lesson plans, delivered lectures, and guided students in technical projects. 
 Used digital tools for coursework and communication, strengthening technical and 
organiza
------------------------------


asking for target job title

In [10]:
# Get job title from user
print("=" * 50)
print("🎯 RAG CV Enhancement Agent")
print("=" * 50)

job_title = input("What job are you targeting? ")

print(f"\n✅ Got it! Enhancing your CV for: {job_title}")

🎯 RAG CV Enhancement Agent
What job are you targeting? junior data analyst 

✅ Got it! Enhancing your CV for: junior data analyst 


setting up LLM API

In [12]:
from google.colab import userdata

# Get your Groq key from secrets
groq_api_key = userdata.get('GroqKey')
llm = ChatGroq(api_key=groq_api_key, model="llama-3.1-8b-instant")

RAG enhancement chain

In [19]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# RAG Enhancement Prompt
rag_enhance_prompt = ChatPromptTemplate.from_template("""
You are a professional CV enhancer.

Context from CV and job description:
{context}

Target Job: {job_title}
Previous feedback: {feedback}

STRICT RULES — you MUST follow these:
- ONLY use information that exists in the context
- NEVER invent or assume any achievements, numbers or experiences
- NEVER add fake metrics like "increased by 45%" unless it's in the context
- If information is missing — improve the language only, don't add fake data
- Only enhance the WRITING STYLE not the CONTENT

Write the enhanced CV:
""")

rag_enhance_chain = rag_enhance_prompt | llm | StrOutputParser()

print("✅ RAG Enhancement Chain ready!")
# Review Prompt
review_prompt = ChatPromptTemplate.from_template("""
You are a strict professional CV reviewer.

Review this CV considering:
Length Check: {length_feedback}
Sections Check: {sections_feedback}

CV to review:
{cv}

Reply in EXACTLY this format:
SCORE: [number between 1-10]
FEEDBACK: [specific details on what to improve]
""")

review_chain = review_prompt | llm | StrOutputParser()

print("✅ Chains ready!")





✅ RAG Enhancement Chain ready!
✅ Chains ready!


making helper tools


In [15]:
import re
from langchain.tools import tool

@tool
def check_cv_length(cv: str) -> str:
    """Check if CV length is appropriate. Ideal is 300-800 words."""
    word_count = len(cv.split())
    if word_count < 300:
        return f"❌ Too short! Only {word_count} words. Needs more detail."
    elif word_count > 800:
        return f"❌ Too long! {word_count} words. Should be under 800."
    else:
        return f"✅ Perfect length! {word_count} words."

@tool
def check_sections(cv: str) -> str:
    """Check if CV has all required sections"""
    cv_lower = cv.lower()
    required_sections = {
        "Professional Summary": "professional summary",
        "Work Experience": "work experience",
        "Skills": "skills",
        "Education": "education"
    }
    missing = []
    found = []
    for section_name, keyword in required_sections.items():
        if re.search(r'\b' + re.escape(keyword) + r'\b:?', cv_lower):
            found.append(section_name)
        else:
            missing.append(section_name)
    if missing:
        return f"❌ Missing: {', '.join(missing)}\n✅ Found: {', '.join(found)}"
    else:
        return f"✅ All sections found: {', '.join(found)}"

print("✅ Tools ready!")

✅ Tools ready!


helper Functions

In [16]:
def extract_score(review_text):
    try:
        for line in review_text.split("\n"):
            if "SCORE:" in line:
                score = line.split("SCORE:")[1].strip()
                return int(''.join(filter(str.isdigit, score)))
    except:
        return 0
    return 0

def extract_feedback(review_text):
    try:
        return review_text.split("FEEDBACK:")[1].strip()
    except:
        return ""

print("✅ Helper functions ready!")

✅ Helper functions ready!


setting up cv agent

In [24]:
def rag_cv_agent(job_title, retriever, original_cv):
    feedback = "None - first attempt"
    attempt = 1
    max_attempts = 4
    scores = []

    while attempt <= max_attempts:
        print(f"\n🔄 Attempt {attempt}...")

        # RAG - Retrieve relevant context
        print("🔍 Retrieving relevant context...")
        query = f"skills and experience required for {job_title}"
        retrieved_docs = retriever.invoke(query)
        context = "\n\n".join([doc.page_content for doc in retrieved_docs])
        print(f"📄 Retrieved {len(retrieved_docs)} relevant chunks!")

        # Enhance CV
        print("✍️ Enhancing CV...")
        cv = rag_enhance_chain.invoke({
            "context": context,
            "job_title": job_title,
            "feedback": feedback
        })

        # Fix 2 - Check for hallucinations
        print("🔎 Checking for fake information...")
        hallucination_check = check_hallucination.invoke({
            "original_cv_and_enhanced": f"{original_cv}|||{cv}"
        })
        print(f"🔎 Hallucination check: {hallucination_check}")

        # If hallucination found - add to feedback
        if "HALLUCINATION: YES" in hallucination_check:
            fake_content = hallucination_check.split("FAKE_CONTENT:")[1].strip()
            feedback = f"IMPORTANT: Remove this fake content: {fake_content}. Only use real information from the original CV!"
            print("⚠️ Fake content detected! Will fix in next attempt...")
            attempt += 1
            continue

        # Run tools
        print("🔧 Running tools...")
        length_feedback = check_cv_length.invoke({"cv": cv})
        sections_feedback = check_sections.invoke({"cv": cv})
        print(f"📏 {length_feedback}")
        print(f"📋 {sections_feedback}")

        # Review CV
        print("📊 Reviewing...")
        review = review_chain.invoke({
            "cv": cv,
            "length_feedback": length_feedback,
            "sections_feedback": sections_feedback
        })

        score = extract_score(review)
        feedback = extract_feedback(review)
        scores.append(score)

        print(f"⭐ Score: {score}/10")

        if score >= 8:
            print(f"\n✅ CV approved with score {score}/10!")
            print(f"📈 Progress: {' → '.join(map(str, scores))}")
            return cv
        else:
            print(f"🔄 Improving based on feedback...")
            attempt += 1

    print(f"\n⚠️ Max attempts reached!")
    print(f"📈 Progress: {' → '.join(map(str, scores))}")
    return cv

In [26]:
from groq import Groq

@tool
def check_hallucination(original_cv_and_enhanced: str) -> str:
    """Check if enhanced CV contains information not in original CV"""
    parts = original_cv_and_enhanced.split("|||")
    original = parts[0]
    enhanced = parts[1]

    # Initialize the Groq client here
    client = Groq(api_key=groq_api_key)

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": """Compare the original and enhanced CV.
Check if the enhanced CV contains ANY information, numbers, or achievements
that are NOT in the original CV.
Reply in EXACTLY this format:
HALLUCINATION: YES or NO
FAKE_CONTENT: [list what was added that wasn't in original, or 'None']"""},
            {"role": "user", "content": f"""
Original CV:
{original}

Enhanced CV:
{enhanced}
"""}
        ]
    )
    return response.choices[0].message.content

In [27]:
result = rag_cv_agent(job_title, retriever, cv_text)

print("\n" + "="*50)
print("📄 FINAL ENHANCED CV:")
print("="*50)
print(result)


🔄 Attempt 1...
🔍 Retrieving relevant context...
📄 Retrieved 3 relevant chunks!
✍️ Enhancing CV...
🔎 Checking for fake information...
🔎 Hallucination check: HALLUCINATION: NO
FAKE_CONTENT: 
- Phone: +92 324 9752902 (repeated in a different format)
- Changed format and wording of some sections (e.g., "Contact Information" instead of simply listing the details, changed tense in "Professional Summary")
- Removed "Currently in Kazakhstan" from a separate line and placed it in the "Contact Information" section
- Removed "Data Analysis Project" from the "Certifications & Projects" section, and only kept the Google Data Analytics Professional Certificate
- Removed "Python (basic) | MS Office (Word, Excel, PowerPoint) | Communication & Reporting | Problem-Solving & Time Management" from the "Core Skills" section, changed wording to "Problem-Solving & Time Management"
🔧 Running tools...
📏 ❌ Too short! Only 169 words. Needs more detail.
📋 ❌ Missing: Work Experience
✅ Found: Professional Summary,